# Analysis of Drinking Waterpoints in Kano, Nigeria
> Note: This notebook requires the local environment dependencies listed in our [requirements.txt] (requirements.txt) file. Use this file to install the required packages in a virtual environment.

> To excecute OpenRouteService functions, it is required to install the [library dependencies](https://github.com/GIScience/openrouteservice-examples#local-installation). You should either have an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local ORS environment to complete the analysis.

The model concepts and processes are described in our documentation. The [Dataset-interpretability](https://github.com/urbanbigdatacentre/ideamaps-models/blob/a4084fb650424ac575941cdacb71421aa882bae4/models/emergency-maternal-care/kano/dataset-interpretability.md) file describes the rationale behind this model.

## Workflow:
The notebook is divided into the following sections:

1. Initial Setup
2. Data Preparation
3. Travel time estimates
4. Two-step floating catchment area (2SFCA) analysis
5. Results

## 1. Initial Setup

## Setting up the virtual environment

```bash
# Create a new virtual environment
# It is recommended to create this virtual environment in the scripts folder
python -m venv .venv

# Activate the virtual environment
source .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [3]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

## Preprocessing
In this study, users first requested an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.

### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# %%
# Read the api key from the .env file
from dotenv import load_dotenv
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
    # Insert R code from the local ORS service
```

For the drinking‑water model in Kano State, we began with 684 waterpoint records from the [Donate Water initiative](https://donatewater.ng), a community‑driven platform that combines crowdsourcing and data analytics to map and monitor water facilities across Nigeria. In [QGIS](https://qgis.org/), we removed six points located more than 50 km from their nearest neighbours, yielding 678 non‑outlier records. We then applied a second round of quality‑control filters—excluding records with “I don’t know” values for water source, ownership, or cost, as well as facilities listed as abandoned or non‑functional—to arrive at our final, validated dataset of 456 waterpoints.

Boundary Framework
When these 678 non‑outlier points were overlaid on the official Kano Local Government Area (Admin Level 2) boundary [Humanitarian Data Exchange, published 25 Nov 2015](https://data.humdata.org/) , 56 (8.3 %) lay just outside the jurisdictional line, clustering along major roads. To ensure our coverage metrics reflected the true operational footprint, we hand‑digitized a bespoke polygon in QGIS that fully encloses all 678 non‑outlier points. After final cleaning, all 456 validated waterpoints fall within this custom service‑area boundary.

Usage in Analysis
From that point onward, every spatial query, point‑density calculation, and service‑area model was executed strictly within our hand‑digitized boundary. By anchoring the analysis to the actual distribution of cleaned, functional waterpoints, we ensure that our density and coverage metrics faithfully represent on‑the‑ground service provision—avoiding the underestimation that would result from forcing the data into administrative lines misaligned with real‑world facility locations.

Data Sources
* [Waterpoint facilities](https://donatewater.ng): generated 12 Oct 2023; accessed 15 Feb 2025 
* [District boundaries](https://data.humdata.org/dataset/nigeria-admin-level-2): Humanitarian Data Exchange, published 25 Nov 2015; accessed 10 Jan 2024
* [World Population Data](https://hub.worldpop.org/geodata/summary?id=28031): generated 2020-02-01; accessed 15 july 2025
* Custom service‑area boundary: Hand‑digitized in QGIS to encompass all non‑outlier waterpoints (2025)

In [ ]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Kano/data-inputs/'                # Current folder
data_temp = '../Kano/data-temp/'       # Goes up one folder, then into data-temp
model_outputs = '../Kano/model-output/'             # Goes up one folder to scripts/Lagos

## Data Collection

### 1.1 Drinking waterpoints for Kano

In [4]:
# Read the drinking water points data
# Ensure the file path is correct and the file exists
drinking_waterpoints = gpd.read_file(data_inputs + 'waterpoints_kano.geojson')

In [5]:
drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,waterSourc,waterState,waterType,Source 2,buffer_m,geometry
0,1,"Dec 13, 2023 @ 11:25:22.000",I don't know,Within 1-5 mins of walking,57.29836,11.96169,8.251364,No,No,Publicly Owned,"Yes, it is fully functional and still in use",Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25131 11.96166)
1,2,"Dec 13, 2023 @ 11:27:47.000",I don't know,Within 1-5 mins of walking,42.92975,11.96487,8.249533,No,No,Publicly Owned,"Yes, it is fully functional and still in use",Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25111 11.96247)
2,3,"Oct 18, 2023 @ 08:21:39.000",I don't know,Within 5-10 mins of walking,176.9393,11.96881,8.249900,No,No,Publicly Owned,"Yes, it is fully functional and still in use",Clear,Hand-dug well,Donate_Water,500.0,POINT (8.25032 11.96736)
3,4,"Dec 11, 2023 @ 08:48:07.000",I don't know,Within 1-5 mins of walking,27.49738,11.96970,8.250319,Yes,Yes,Publicly Owned,"No, it is not functional and not in use",Clear,Borehole with handpump,Donate_Water,250.0,POINT (8.24948 11.96991)
4,5,"Dec 11, 2023 @ 12:26:22.000",I don't know,Within 5-10 mins of walking,76.66726,11.97814,8.265445,No,No,Publicly Owned,"Yes, it is fully functional and still in use",Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.26036 11.97622)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673,680,"Apr 30, 2024 @ 04:36:50.000",I don't know,Within 1-5 mins of walking,65.52058,11.97458,8.626487,No,No,Privately Owned,"Yes, it is fully functional and still in use",Clear,Borehole with tap,Donate_Water,250.0,POINT (8.62645 11.97448)
674,681,"Oct 29, 2023 @ 07:12:22.000",I don't know,Within 5-10 mins of walking,158.5212,11.97388,8.630894,Don't know,No,Publicly Owned,"Yes, it is fully functional and still in use",Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.62835 11.97452)
675,682,"Dec 3, 2023 @ 06:57:01.000",I don't know,Within 1-5 mins of walking,27.13257,11.96971,8.628362,Yes,Yes,Publicly Owned,"No, it is not functional and not in use",Clear,Hand-dug well,Donate_Water,250.0,POINT (8.62833 11.96981)
676,683,"Dec 3, 2023 @ 07:38:23.000",I don't know,Within 5-10 mins of walking,211.5117,11.97468,8.625813,Yes,No,Privately Owned,"Yes, it is fully functional and still in use",Clear,Other,Donate_Water,500.0,POINT (8.62699 11.97468)


In [6]:
# Define unwanted categories
unwanted_sources = ["No, it is not functional and not in use", "No, it is abandoned", "I don't know"]

# Filter them out
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterSourc'].isin(unwanted_sources)]


In [7]:
# Map the water source valuess to better descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterSourc'] = (
    drinking_waterpoints['waterSourc']
    .astype(str)
    .str.strip()
    .map({
        'Yes, it is fully functional and still in use': 'Fully Functional',
        'Yes, it is partially functional and still in use': 'Partially Functional'
    })
)

/Users/o.shonowo.1/Library/CloudStorage/OneDrive-UniversityofGlasgow/Drinking Water Model/.venv/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [8]:
# Define unwanted categories
unwanted_sources1 = ["Don't know"]

# Filter them out
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterPubli'].isin(unwanted_sources1)]
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterFree'].isin(unwanted_sources1)]


In [9]:
drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,waterSourc,waterState,waterType,Source 2,buffer_m,geometry
0,1,"Dec 13, 2023 @ 11:25:22.000",I don't know,Within 1-5 mins of walking,57.29836,11.96169,8.251364,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25131 11.96166)
1,2,"Dec 13, 2023 @ 11:27:47.000",I don't know,Within 1-5 mins of walking,42.92975,11.96487,8.249533,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25111 11.96247)
2,3,"Oct 18, 2023 @ 08:21:39.000",I don't know,Within 5-10 mins of walking,176.9393,11.96881,8.249900,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,500.0,POINT (8.25032 11.96736)
4,5,"Dec 11, 2023 @ 12:26:22.000",I don't know,Within 5-10 mins of walking,76.66726,11.97814,8.265445,No,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.26036 11.97622)
6,7,"Dec 12, 2023 @ 11:57:09.000",I don't know,Within 1-5 mins of walking,64.95459,11.94213,8.257465,No,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,250.0,POINT (8.25731 11.94212)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,679,"Dec 3, 2023 @ 06:42:02.000",I don't know,Within 5-10 mins of walking,67.63028,11.97268,8.626462,No,No,Publicly Owned,Fully Functional,Dirty,Hand-dug well,Donate_Water,500.0,POINT (8.62649 11.9726)
673,680,"Apr 30, 2024 @ 04:36:50.000",I don't know,Within 1-5 mins of walking,65.52058,11.97458,8.626487,No,No,Privately Owned,Fully Functional,Clear,Borehole with tap,Donate_Water,250.0,POINT (8.62645 11.97448)
674,681,"Oct 29, 2023 @ 07:12:22.000",I don't know,Within 5-10 mins of walking,158.5212,11.97388,8.630894,Don't know,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.62835 11.97452)
676,683,"Dec 3, 2023 @ 07:38:23.000",I don't know,Within 5-10 mins of walking,211.5117,11.97468,8.625813,Yes,No,Privately Owned,Fully Functional,Clear,Other,Donate_Water,500.0,POINT (8.62699 11.97468)


In [12]:
# Define mapping from waterState to waterDrink for "I don't know"
state_to_drink = {
    "Clear": "Yes",
    "Dirty": "No",
    "Currently no water available": "No",
    "Muddy": "No",
    "Coloured": "No"
}

# Copy the original waterDrink column
drinking_waterpoints['waterDrink_estimated'] = drinking_waterpoints['waterDrink']

# Only update values in waterDrink_estimated where waterDrink is "I don't know"
drinking_waterpoints.loc[drinking_waterpoints['waterDrink'] == "Don't know", 'waterDrink_estimated'] = drinking_waterpoints.loc[
    drinking_waterpoints['waterDrink'] == "Don't know", 'waterState'
].map(state_to_drink)

In [13]:
drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,waterSourc,waterState,waterType,Source 2,buffer_m,geometry,waterDrink_estimated
0,1,"Dec 13, 2023 @ 11:25:22.000",I don't know,Within 1-5 mins of walking,57.29836,11.96169,8.251364,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25131 11.96166),No
1,2,"Dec 13, 2023 @ 11:27:47.000",I don't know,Within 1-5 mins of walking,42.92975,11.96487,8.249533,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25111 11.96247),No
2,3,"Oct 18, 2023 @ 08:21:39.000",I don't know,Within 5-10 mins of walking,176.9393,11.96881,8.249900,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,500.0,POINT (8.25032 11.96736),No
4,5,"Dec 11, 2023 @ 12:26:22.000",I don't know,Within 5-10 mins of walking,76.66726,11.97814,8.265445,No,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.26036 11.97622),No
6,7,"Dec 12, 2023 @ 11:57:09.000",I don't know,Within 1-5 mins of walking,64.95459,11.94213,8.257465,No,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,250.0,POINT (8.25731 11.94212),No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,679,"Dec 3, 2023 @ 06:42:02.000",I don't know,Within 5-10 mins of walking,67.63028,11.97268,8.626462,No,No,Publicly Owned,Fully Functional,Dirty,Hand-dug well,Donate_Water,500.0,POINT (8.62649 11.9726),No
673,680,"Apr 30, 2024 @ 04:36:50.000",I don't know,Within 1-5 mins of walking,65.52058,11.97458,8.626487,No,No,Privately Owned,Fully Functional,Clear,Borehole with tap,Donate_Water,250.0,POINT (8.62645 11.97448),No
674,681,"Oct 29, 2023 @ 07:12:22.000",I don't know,Within 5-10 mins of walking,158.5212,11.97388,8.630894,Don't know,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.62835 11.97452),Yes
676,683,"Dec 3, 2023 @ 07:38:23.000",I don't know,Within 5-10 mins of walking,211.5117,11.97468,8.625813,Yes,No,Privately Owned,Fully Functional,Clear,Other,Donate_Water,500.0,POINT (8.62699 11.97468),Yes


In [14]:
# Define user-specified improved water source types
improved = [
    'piped-borne water',
    'borehole with hand pump',
    'borehole with tap',
    'water vendors',
    'spring'
]

# Create a new column 'watertype_revised' with classification
drinking_waterpoints['watertype_revised'] = drinking_waterpoints['waterType'].apply(
    lambda x: 'Improved' if isinstance(x, str) and x.strip().lower() in improved else 'Unimproved'
)

drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,waterSourc,waterState,waterType,Source 2,buffer_m,geometry,waterDrink_estimated,watertype_revised
0,1,"Dec 13, 2023 @ 11:25:22.000",I don't know,Within 1-5 mins of walking,57.29836,11.96169,8.251364,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25131 11.96166),No,Unimproved
1,2,"Dec 13, 2023 @ 11:27:47.000",I don't know,Within 1-5 mins of walking,42.92975,11.96487,8.249533,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25111 11.96247),No,Unimproved
2,3,"Oct 18, 2023 @ 08:21:39.000",I don't know,Within 5-10 mins of walking,176.9393,11.96881,8.249900,No,No,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,500.0,POINT (8.25032 11.96736),No,Unimproved
4,5,"Dec 11, 2023 @ 12:26:22.000",I don't know,Within 5-10 mins of walking,76.66726,11.97814,8.265445,No,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.26036 11.97622),No,Unimproved
6,7,"Dec 12, 2023 @ 11:57:09.000",I don't know,Within 1-5 mins of walking,64.95459,11.94213,8.257465,No,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,250.0,POINT (8.25731 11.94212),No,Unimproved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,679,"Dec 3, 2023 @ 06:42:02.000",I don't know,Within 5-10 mins of walking,67.63028,11.97268,8.626462,No,No,Publicly Owned,Fully Functional,Dirty,Hand-dug well,Donate_Water,500.0,POINT (8.62649 11.9726),No,Unimproved
673,680,"Apr 30, 2024 @ 04:36:50.000",I don't know,Within 1-5 mins of walking,65.52058,11.97458,8.626487,No,No,Privately Owned,Fully Functional,Clear,Borehole with tap,Donate_Water,250.0,POINT (8.62645 11.97448),No,Improved
674,681,"Oct 29, 2023 @ 07:12:22.000",I don't know,Within 5-10 mins of walking,158.5212,11.97388,8.630894,Don't know,No,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.62835 11.97452),Yes,Unimproved
676,683,"Dec 3, 2023 @ 07:38:23.000",I don't know,Within 5-10 mins of walking,211.5117,11.97468,8.625813,Yes,No,Privately Owned,Fully Functional,Clear,Other,Donate_Water,500.0,POINT (8.62699 11.97468),Yes,Unimproved


In [15]:
# map the waterDrink_estimated values to more descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterDrink_estimated'] = (
    drinking_waterpoints['waterDrink_estimated']
    .astype(str)
    .str.strip()
    .map({
        'Yes': 'Drinkable',
        'No': 'Non drinkable'
    })
)

In [16]:
# map the waterFree values to more descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterFree'] = (
    drinking_waterpoints['waterFree']
    .astype(str)
    .str.strip()
    .map({
        'Yes': 'Free',
        'No': 'Non Free'
    })
)

In [17]:
drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,waterSourc,waterState,waterType,Source 2,buffer_m,geometry,waterDrink_estimated,watertype_revised
0,1,"Dec 13, 2023 @ 11:25:22.000",I don't know,Within 1-5 mins of walking,57.29836,11.96169,8.251364,No,Non Free,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25131 11.96166),Non drinkable,Unimproved
1,2,"Dec 13, 2023 @ 11:27:47.000",I don't know,Within 1-5 mins of walking,42.92975,11.96487,8.249533,No,Non Free,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,250.0,POINT (8.25111 11.96247),Non drinkable,Unimproved
2,3,"Oct 18, 2023 @ 08:21:39.000",I don't know,Within 5-10 mins of walking,176.9393,11.96881,8.249900,No,Non Free,Publicly Owned,Fully Functional,Clear,Hand-dug well,Donate_Water,500.0,POINT (8.25032 11.96736),Non drinkable,Unimproved
4,5,"Dec 11, 2023 @ 12:26:22.000",I don't know,Within 5-10 mins of walking,76.66726,11.97814,8.265445,No,Non Free,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.26036 11.97622),Non drinkable,Unimproved
6,7,"Dec 12, 2023 @ 11:57:09.000",I don't know,Within 1-5 mins of walking,64.95459,11.94213,8.257465,No,Non Free,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,250.0,POINT (8.25731 11.94212),Non drinkable,Unimproved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,679,"Dec 3, 2023 @ 06:42:02.000",I don't know,Within 5-10 mins of walking,67.63028,11.97268,8.626462,No,Non Free,Publicly Owned,Fully Functional,Dirty,Hand-dug well,Donate_Water,500.0,POINT (8.62649 11.9726),Non drinkable,Unimproved
673,680,"Apr 30, 2024 @ 04:36:50.000",I don't know,Within 1-5 mins of walking,65.52058,11.97458,8.626487,No,Non Free,Privately Owned,Fully Functional,Clear,Borehole with tap,Donate_Water,250.0,POINT (8.62645 11.97448),Non drinkable,Improved
674,681,"Oct 29, 2023 @ 07:12:22.000",I don't know,Within 5-10 mins of walking,158.5212,11.97388,8.630894,Don't know,Non Free,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.62835 11.97452),Drinkable,Unimproved
676,683,"Dec 3, 2023 @ 07:38:23.000",I don't know,Within 5-10 mins of walking,211.5117,11.97468,8.625813,Yes,Non Free,Privately Owned,Fully Functional,Clear,Other,Donate_Water,500.0,POINT (8.62699 11.97468),Drinkable,Unimproved


In [18]:
# 1. Load study area
study_area = gpd.read_file(data_inputs + "grid-boundary-kano.gpkg")

# 2. Ensure CRS match
if study_area.crs is not None and drinking_waterpoints.crs != study_area.crs:
    drinking_waterpoints = drinking_waterpoints.to_crs(study_area.crs)

# 3. Clip points to study area
drinking_waterpoints = gpd.clip(drinking_waterpoints, study_area)

In [19]:
# Define conditions for categorizing water points into limited, optimal, and moderate water categories
# Ensure the conditions are correctly defined based on the data
conditions = [
    drinking_waterpoints['waterDrink_estimated'] == 'Non drinkable',
    (drinking_waterpoints['waterSourc'] == 'Fully Functional') & (drinking_waterpoints['watertype_revised'] == 'Improved')
]

categories = ['Limited Water', 'Optimal Water']

drinking_waterpoints['category'] = np.select(conditions, categories, default='Moderate Water')

/Users/o.shonowo.1/Library/CloudStorage/OneDrive-UniversityofGlasgow/Drinking Water Model/.venv/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [20]:
# Display the counts of each category
# Ensure the counts are correctly calculated
category_counts = drinking_waterpoints['category'].value_counts()
category_counts

category
Limited Water     223
Moderate Water    116
Optimal Water      84
Name: count, dtype: int64

In [21]:
drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,waterSourc,waterState,waterType,Source 2,buffer_m,geometry,waterDrink_estimated,watertype_revised,category
67,68,"Oct 21, 2023 @ 12:48:40.000",I don't know,Within 10-20 mins of walking,178.5332,11.87206,8.374188,No,Non Free,Publicly Owned,Fully Functional,Dirty,Borehole with handpump,Donate_Water,1000.0,POINT (8.37426 11.87205),Non drinkable,Unimproved,Limited Water
72,73,"Oct 21, 2023 @ 10:40:28.000",I don't know,Within 20-30 mins of walking,133.98,11.87846,8.355279,No,Non Free,Publicly Owned,Fully Functional,Muddy,River,Donate_Water,1500.0,POINT (8.35524 11.87854),Non drinkable,Unimproved,Limited Water
49,50,"Oct 25, 2023 @ 09:50:41.000",I don't know,Within 5-10 mins of walking,125.6432,11.87896,8.354836,Don't know,Non Free,Publicly Owned,Partially Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.35496 11.87923),Drinkable,Unimproved,Moderate Water
73,74,"Dec 12, 2023 @ 10:41:37.000",I don't know,Within 5-10 mins of walking,41.98828,11.89218,8.367886,No,Non Free,Publicly Owned,Fully Functional,Clear,Borehole with handpump,Donate_Water,500.0,POINT (8.37017 11.89167),Non drinkable,Unimproved,Limited Water
63,64,"Oct 16, 2023 @ 09:14:04.000",I don't know,Within 20-30 mins of walking,48.17432,11.89170,8.370212,No,Non Free,Publicly Owned,Partially Functional,Dirty,Hand-dug well,Donate_Water,1500.0,POINT (8.37083 11.892),Non drinkable,Unimproved,Limited Water
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,204,"Dec 7, 2023 @ 01:51:40.000","No, there's usually no queue",Within 1-5 mins of walking,96.39276,12.00014,8.525556,Yes,Non Free,Publicly Owned,Partially Functional,Clear,Piped-borne water,Donate_Water,250.0,POINT (8.5256 12.00015),Drinkable,Improved,Moderate Water
209,216,"Oct 26, 2023 @ 06:35:26.000",I don't know,Within 10-20 mins of walking,80.86514,12.00004,8.527484,Don't know,Free,Privately Owned,Fully Functional,Clear,Other,Donate_Water,1000.0,POINT (8.52749 12.0002),Drinkable,Unimproved,Moderate Water
196,203,"Oct 26, 2023 @ 06:32:17.000",I don't know,Within 5-10 mins of walking,94.79898,12.00016,8.525509,Don't know,Non Free,Publicly Owned,Fully Functional,Clear,Piped-borne water,Donate_Water,500.0,POINT (8.52569 12.00025),Drinkable,Improved,Optimal Water
184,191,"Oct 28, 2023 @ 02:22:55.000",Sometimes it depends on the time of day\t,Within 1-5 mins of walking,86.19946,12.00031,8.526118,Yes,Non Free,Publicly Owned,Fully Functional,Clear,Piped-borne water,Donate_Water,250.0,POINT (8.52607 12.00051),Drinkable,Improved,Optimal Water


In [ ]:
# Assign a unique identifier to each water point
# Ensure the identifier is unique and sequential
drinking_waterpoints['waterpoint_id'] = range (1, len(drinking_waterpoints) + 1)

In [ ]:
drinking_waterpoints

,fid,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterFree,waterPubli,...,index_right,dep_bin,latitude,longitude,lon_min,lat_min,lon_max,lat_max,category,waterpoint_id
159,161,"Dec 16, 2023 @ 12:25:42.000",I don't know,Within 1-5 mins of walking,88.56604,11.88002,8.681396,Yes,Non Free,Publicly Owned,...,149682,0,11.879892,8.680655,8.680141,11.879484,8.681169,11.880300,Optimal Water,1
164,169,"Dec 16, 2023 @ 11:57:28.000",I don't know,Within 1-5 mins of walking,198.409,11.88038,8.678519,No,Non Free,Publicly Owned,...,149408,0,11.879892,8.679644,8.679130,11.879484,8.680157,11.880300,Limited Water,2
158,160,"Dec 16, 2023 @ 11:49:55.000",I don't know,Within 1-5 mins of walking,78.81152,11.87966,8.680004,No,Non Free,Publicly Owned,...,149682,0,11.879892,8.680655,8.680141,11.879484,8.681169,11.880300,Limited Water,3
165,170,"Dec 16, 2023 @ 11:44:42.000",I don't know,Within 1-5 mins of walking,68.41797,11.88016,8.681887,No,Free,Privately Owned,...,139693,0,11.916591,8.644972,8.644459,11.916183,8.645486,11.916998,Limited Water,4
613,620,"May 1, 2024 @ 06:52:14.000",I don't know,Within 5-10 mins of walking,51.32434,11.95742,8.618676,No,Free,Privately Owned,...,131575,0,11.957369,8.618468,8.617955,11.956961,8.618982,11.957777,Limited Water,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,358,"Dec 17, 2023 @ 09:55:58.000",I don't know,Within 1-5 mins of walking,57.08533,11.98929,8.575065,No,Non Free,Publicly Owned,...,117159,0,11.989993,8.574607,8.574093,11.989585,8.575121,11.990401,Limited Water,419
312,319,"Oct 29, 2023 @ 12:33:06.000",I don't know,Within 5-10 mins of walking,83.89014,11.99004,8.574145,No,Non Free,Publicly Owned,...,116838,0,11.989993,8.573596,8.573082,11.989585,8.574109,11.990401,Limited Water,420
336,343,"Dec 17, 2023 @ 09:49:12.000",I don't know,Within 1-5 mins of walking,37.14697,11.99001,8.574184,No,Non Free,Publicly Owned,...,116838,0,11.989993,8.573596,8.573082,11.989585,8.574109,11.990401,Limited Water,421
335,342,"Oct 29, 2023 @ 12:35:16.000",I don't know,Within 5-10 mins of walking,57.15479,11.98968,8.574209,No,Non Free,Publicly Owned,...,117159,0,11.989993,8.574607,8.574093,11.989585,8.575121,11.990401,Limited Water,422


In [ ]:
# Write to GeoJSON
drinking_waterpoints.to_file(data_inputs + 'Kano_DW.geojson', driver='GeoJSON')

# Population Grid Data

In [ ]:
study_area = gpd.read_file(data_inputs + "grid-boundary-kano.gpkg")
population_data = data_inputs + "nga_ppp_2020_UNadj.tif"

In [ ]:
study_area["grid_id"] = range(1, len(study_area) + 1)

In [ ]:
with rasterio.open(population_data) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_31586/1658636788.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


In [ ]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [ ]:
with rasterio.open(data_inputs + 'kano_nga_ppp_2020_UNadj.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

In [ ]:
# 1. Open the population raster
with rasterio.open(data_inputs + "Kano_nga_ppp_2020_UNadj.tif") as src:
    data = src.read(1)
    transform = src.transform
    crs = src.crs

    rows, cols = data.shape

    polygons = []
    values = []

    for row in range(rows):
        for col in range(cols):
            value = data[row, col]
            if value == src.nodata:
                continue  

            x, y = rasterio.transform.xy(transform, row, col, offset='ul')

            pixel_width = transform.a
            pixel_height = -transform.e 

            poly = box(x, y - pixel_height, x + pixel_width, y)
            polygons.append(poly)
            values.append(value)

population_kano = gpd.GeoDataFrame({
    "value": values,
    "geometry": polygons
}, crs=crs)

# 7. Save to GeoPackage (two layers: polygons and centroids)
population_kano.to_file("population_kano.gpkg", driver="GPKG")

In [ ]:
# 5. Calculate centroids
population_kano["centroid"] = population_kano.geometry.centroid

# 6. Create centroid layer (points)
population_centroids = population_kano[["value", "centroid"]].copy()
population_centroids = population_centroids.rename(columns={"centroid": "geometry"})
population_centroids = population_centroids.set_geometry("geometry")
population_centroids.set_crs("EPSG:4326", inplace=True)

# 4. Add unique ID
population_centroids["id"] = range(1, len(population_centroids) + 1)

population_centroids.to_file("population_centroids_kano.gpkg", driver="GPKG")

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_31586/3967691660.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  population_kano["centroid"] = population_kano.geometry.centroid


In [ ]:

population_centroids

,value,geometry,id
0,10.517168,POINT (8.51083 12.24917),1
1,11.943038,POINT (8.51167 12.24917),2
2,12.674385,POINT (8.5125 12.24917),3
3,13.146225,POINT (8.51333 12.24917),4
4,12.696987,POINT (8.51417 12.24917),5
...,...,...,...
198574,4.471054,POINT (8.54917 11.73333),198575
198575,4.331891,POINT (8.55 11.73333),198576
198576,4.178248,POINT (8.55083 11.73333),198577
198577,4.014533,POINT (8.55167 11.73333),198578


In [ ]:
# === Ensure same CRS ===
if study_area.crs != population_centroids.crs:
    population_centroids = population_centroids.to_crs(study_area.crs)

# === STEP 1: Spatial join to assign population values ===
join_gdf = gpd.sjoin(
    population_centroids,
    study_area,
    how="left",
    predicate="within"
)

# Compute mean population for each grid_id (works for 1 or 2 centroids)
pop_by_grid = (
    join_gdf.groupby("grid_id")
    .agg({"value": "mean"})
    .reset_index()
)

# Merge into grid
kano_grid_with_pop = study_area.merge(pop_by_grid, on="grid_id", how="left")

# 5. Identify empty grids (no population assigned)
empty_grids = kano_grid_with_pop[kano_grid_with_pop['value'].isna()].copy()

if not empty_grids.empty:
    # 6. Build KDTree from population centroid coordinates
    pop_points = np.array([(pt.x, pt.y) for pt in population_centroids.geometry])
    pop_tree = cKDTree(pop_points)

    # 7. Get centroids of empty grids
    empty_centroids = np.array([(geom.centroid.x, geom.centroid.y) for geom in empty_grids.geometry])

    # 8. Query nearest population centroid for each empty grid centroid
    dist, idx = pop_tree.query(empty_centroids, k=1)

    # 9. Retrieve population values of nearest centroids
    nearest_values = population_centroids.iloc[idx]['value'].values

    # 10. Assign these values to the empty grids
    empty_grids['value'] = nearest_values
    kano_grid_with_pop.loc[empty_grids.index, 'value'] = empty_grids['value']

# 11. Save the updated grid with population values to a GeoPackage
output_path = data_temp + "kano_grid_pop.gpkg"
kano_grid_with_pop.to_file(output_path, driver="GPKG")


In [ ]:
kano_grid_pop_centroids = kano_grid_with_pop.copy()
kano_grid_pop_centroids['centroid'] = kano_grid_pop_centroids.geometry.centroid
kano_grid_pop_centroids = kano_grid_pop_centroids.set_geometry('centroid')

kano_grid_pop_centroids = kano_grid_pop_centroids.drop(columns='geometry')

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_31586/348819975.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  kano_grid_pop_centroids['centroid'] = kano_grid_pop_centroids.geometry.centroid


In [ ]:


kano_grid_pop_centroids.to_file(
    data_temp + 'kano_grid_pop_centroids.geojson',
    driver='GeoJSON'
)


# OD matrix

In [ ]:
# Read the Kano OD matrix
# Ensure the file path is correct and the file exists
Kano_OD_Matrix = pd.read_csv(data_inputs + 'kano_odmatrix.csv')
Kano_OD_Matrix

,origin_id,destination_id,duration_seconds,distance_km,duration_minutes
0,1,1,15513.62,21.55,258.560333
1,1,2,15127.58,21.01,252.126333
2,1,3,14753.15,20.49,245.885833
3,1,4,13975.40,19.41,232.923333
4,1,5,24649.38,34.24,410.823000
...,...,...,...,...,...
70750975,167260,419,16616.86,23.08,276.947667
70750976,167260,420,16488.24,22.90,274.804000
70750977,167260,421,16513.82,22.94,275.230333
70750978,167260,422,16461.09,22.86,274.351500


In [ ]:
# Filter rows where duration_seconds <= 3600
Kano_OD_Matrix_quota = Kano_OD_Matrix[Kano_OD_Matrix['duration_seconds'] <= 3600]  # 1 hour

Kano_OD_Matrix_quota

,origin_id,destination_id,duration_seconds,distance_km,duration_minutes
8478,21,19,1556.61,2.16,25.943500
8479,21,20,1194.43,1.66,19.907167
8480,21,21,1431.41,1.99,23.856833
8481,21,22,1572.50,2.18,26.208333
8482,21,23,1457.69,2.02,24.294833
...,...,...,...,...,...
67679250,159999,97,3468.11,4.82,57.801833
67679251,159999,98,3501.86,4.86,58.364333
67679254,159999,101,3538.80,4.92,58.980000
67793883,160270,97,3555.89,4.94,59.264833


In [ ]:
# Merge the Kano OD matrix with the population centroids grid cells
pop_centroid_matrix_quota = pd.merge(Kano_OD_Matrix_quota, kano_grid_with_pop[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','value', 'geometry']],
                             left_on='origin_id', right_on='grid_id', how='left')

In [ ]:
# Display the merged population centroid matrix
# Ensure the data is correctly merged and displayed
pop_centroid_matrix_quota

,origin_id,destination_id,duration_seconds,distance_km,duration_minutes,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,value,geometry
0,21,19,1556.61,2.16,25.943500,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
1,21,20,1194.43,1.66,19.907167,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
2,21,21,1431.41,1.99,23.856833,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
3,21,22,1572.50,2.18,26.208333,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
4,21,23,1457.69,2.02,24.294833,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1814970,159999,97,3468.11,4.82,57.801833,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8..."
1814971,159999,98,3501.86,4.86,58.364333,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8..."
1814972,159999,101,3538.80,4.92,58.980000,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8..."
1814973,160270,97,3555.89,4.94,59.264833,160270,8.718710,11.861136,8.718196,11.860728,8.719224,11.861544,2.151011,"POLYGON ((8.71821 11.86154, 8.71922 11.86154, ..."


In [ ]:
# Rename columns for clarity
pop_centroid_matrix_quota = pop_centroid_matrix_quota.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "destination_id": "dwp_id",
    "value": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min", "origin_lat_min", "origin_lon_max", "origin_lat_max", "population", "geometry", "dwp_id", "duration_seconds","duration_minutes", "distance_km"]
pop_centroid_matrix_quota = pop_centroid_matrix_quota[columns_to_keep]

In [ ]:
pop_centroid_matrix_quota

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_minutes,distance_km
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",19,1556.61,25.943500,2.16
1,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",20,1194.43,19.907167,1.66
2,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",21,1431.41,23.856833,1.99
3,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",22,1572.50,26.208333,2.18
4,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",23,1457.69,24.294833,2.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1814970,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",97,3468.11,57.801833,4.82
1814971,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",98,3501.86,58.364333,4.86
1814972,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",101,3538.80,58.980000,4.92
1814973,160270,8.718710,11.861136,8.718196,11.860728,8.719224,11.861544,2.151011,"POLYGON ((8.71821 11.86154, 8.71922 11.86154, ...",97,3555.89,59.264833,4.94


In [ ]:
# Convert waterpoint_id to numeric and ensure it is an integer
drinking_waterpoints['waterpoint_id'] = pd.to_numeric(drinking_waterpoints['waterpoint_id'], errors='raise').astype(int)

print(drinking_waterpoints['waterpoint_id'].dtype) 

int64


In [ ]:
print(pop_centroid_matrix_quota['dwp_id'].dtype) 

int64


In [ ]:
# Merge the population centroid matrix with the drinking water points
# Ensure the merge is done correctly and the columns are aligned
# This will add the waterpoint_id and other relevant columns to the population centroid matrix
distances_duration_matrix = pd.merge(pop_centroid_matrix_quota, drinking_waterpoints[['waterpoint_id', 'category', 'userPosi_1', 'userPosi_2']], 
                     left_on='dwp_id', right_on='waterpoint_id', how='left')

In [ ]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_minutes,distance_km,waterpoint_id,category,userPosi_1,userPosi_2
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",19,1556.61,25.943500,2.16,19,Moderate Water,11.97047,8.440383
1,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",20,1194.43,19.907167,1.66,20,Optimal Water,11.97092,8.439520
2,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",21,1431.41,23.856833,1.99,21,Moderate Water,11.97442,8.438909
3,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",22,1572.50,26.208333,2.18,22,Optimal Water,11.97425,8.439657
4,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",23,1457.69,24.294833,2.02,23,Limited Water,11.97093,8.442822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1814970,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",97,3468.11,57.801833,4.82,97,Limited Water,11.87966,8.680004
1814971,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",98,3501.86,58.364333,4.86,98,Optimal Water,11.88002,8.681396
1814972,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",101,3538.80,58.980000,4.92,101,Limited Water,11.88038,8.678519
1814973,160270,8.718710,11.861136,8.718196,11.860728,8.719224,11.861544,2.151011,"POLYGON ((8.71821 11.86154, 8.71922 11.86154, ...",97,3555.89,59.264833,4.94,97,Limited Water,11.87966,8.680004


In [ ]:
distances_duration_matrix = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")
distances_duration_matrix.to_file(data_temp + 'distances_duration_matrix_quota.geojson', driver='GeoJSON')

In [ ]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_minutes,distance_km,waterpoint_id,category,userPosi_1,userPosi_2
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",19,1556.61,25.943500,2.16,19,Moderate Water,11.97047,8.440383
1,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",20,1194.43,19.907167,1.66,20,Optimal Water,11.97092,8.439520
2,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",21,1431.41,23.856833,1.99,21,Moderate Water,11.97442,8.438909
3,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",22,1572.50,26.208333,2.18,22,Optimal Water,11.97425,8.439657
4,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",23,1457.69,24.294833,2.02,23,Limited Water,11.97093,8.442822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1814970,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",97,3468.11,57.801833,4.82,97,Limited Water,11.87966,8.680004
1814971,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",98,3501.86,58.364333,4.86,98,Optimal Water,11.88002,8.681396
1814972,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",101,3538.80,58.980000,4.92,101,Limited Water,11.88038,8.678519
1814973,160270,8.718710,11.861136,8.718196,11.860728,8.719224,11.861544,2.151011,"POLYGON ((8.71821 11.86154, 8.71922 11.86154, ...",97,3555.89,59.264833,4.94,97,Limited Water,11.87966,8.680004


In [ ]:
# Count the number of water points in each category
# Ensure the counts are correctly calculated
category_counts = drinking_waterpoints['category'].value_counts()
print(category_counts)

category
Limited Water     223
Moderate Water    116
Optimal Water      84
Name: count, dtype: int64


In [ ]:
# Display the unique categories in the distances_duration_matrix
print(distances_duration_matrix['category'].unique())

['Moderate Water' 'Optimal Water' 'Limited Water']


In [ ]:
# Define the categories for water points
categories = {
    'Optimal_Water': ['Optimal Water'],
    'Moderate_Water': ['Moderate Water'],
    'Limited_Water': ['Limited Water']
} 

In [ ]:
# step 1: Create subsets of the distances_duration_matrix based on the defined categories
# Ensure the subsets are correctly created and the data is filtered based on the categories
subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['category'].isin(values)
    ]
    for key, values in categories.items()
}

In [ ]:
# Extract subsets for each category
Optimal_Water_subset = subsets['Optimal_Water']
Moderate_Water_subset = subsets['Moderate_Water']
Limited_Water_subset = subsets['Limited_Water']

In [ ]:
# Step 2: Define a function to get the smallest duration_seconds per grid_id for each category
def get_closest(df, n=1):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_minutes')).reset_index(drop=True)


In [ ]:
# If the subsets are already created for each category, we apply the function to each subset:
Optimal_Water_closest = get_closest(Optimal_Water_subset)
Moderate_Water_closest = get_closest(Moderate_Water_subset)
Limited_Water_closest = get_closest(Limited_Water_subset)

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_31586/459179960.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_minutes')).reset_index(drop=True)
/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_31586/459179960.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'd

In [ ]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    Optimal_Water_closest, Moderate_Water_closest, Limited_Water_closest])

In [ ]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_minutes,distance_km,waterpoint_id,category,userPosi_1,userPosi_2
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",24,1142.25,19.037500,1.59,24,Optimal Water,11.96988,8.441043
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,223.528427,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ...",22,308.07,5.134500,0.43,22,Optimal Water,11.97425,8.439657
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,233.314590,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ...",67,111.02,1.850333,0.15,67,Optimal Water,11.96606,8.449456
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,247.544235,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8...",110,933.16,15.552667,1.30,110,Optimal Water,11.97467,8.455998
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,140.231598,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8....",110,2683.72,44.728667,3.73,110,Optimal Water,11.97467,8.455998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43681,159996,8.717747,11.863582,8.717234,11.863175,8.718261,11.863990,4.837119,"POLYGON ((8.71725 11.86399, 8.71826 11.86399, ...",97,3586.59,59.776500,4.98,97,Limited Water,11.87966,8.680004
43682,159997,8.717731,11.862767,8.717217,11.862359,8.718245,11.863175,3.875393,"POLYGON ((8.71723 11.86317, 8.71824 11.86317, ...",97,3553.37,59.222833,4.94,97,Limited Water,11.87966,8.680004
43683,159998,8.717715,11.861951,8.717201,11.861544,8.718229,11.862359,3.067920,"POLYGON ((8.71722 11.86236, 8.71823 11.86236, ...",97,3503.28,58.388000,4.87,97,Limited Water,11.87966,8.680004
43684,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",97,3468.11,57.801833,4.82,97,Limited Water,11.87966,8.680004


In [ ]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "userPosi_2": "dest_lon",
    "userPosi_1": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['dwp_id'])

In [ ]:
# Add geometry to the DataFrame because the OD matrix is a CSV file and does not have geometry information.
# We create a geometry column using the origin coordinates.
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")

In [ ]:
# Ensure the geometry is correctly created from the origin coordinates
gpkg_path = data_temp + 'distances_duration_closest_DW.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_closest_DW", driver="GPKG")

In [ ]:
# Review and remove
origin_dest = gpd.read_file(data_temp + "distances_duration_closest_DW.gpkg", layer="distances_duration_closest_DW")

In [ ]:
origin_dest

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_minutes,distance_km,waterpoint_id,category,dest_lat,dest_lon,geometry
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,1142.25,19.037500,1.59,24,Optimal Water,11.96988,8.441043,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,223.528427,308.07,5.134500,0.43,22,Optimal Water,11.97425,8.439657,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ..."
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,233.314590,111.02,1.850333,0.15,67,Optimal Water,11.96606,8.449456,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ..."
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,247.544235,933.16,15.552667,1.30,110,Optimal Water,11.97467,8.455998,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8..."
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,140.231598,2683.72,44.728667,3.73,110,Optimal Water,11.97467,8.455998,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103550,159996,8.717747,11.863582,8.717234,11.863175,8.718261,11.863990,4.837119,3586.59,59.776500,4.98,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71725 11.86399, 8.71826 11.86399, ..."
103551,159997,8.717731,11.862767,8.717217,11.862359,8.718245,11.863175,3.875393,3553.37,59.222833,4.94,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71723 11.86317, 8.71824 11.86317, ..."
103552,159998,8.717715,11.861951,8.717201,11.861544,8.718229,11.862359,3.067920,3503.28,58.388000,4.87,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71722 11.86236, 8.71823 11.86236, ..."
103553,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,3468.11,57.801833,4.82,97,Limited Water,11.87966,8.680004,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8..."


## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [ ]:
# Function to calculate beta based on the maximum distance and a given W value
from math import *
d = 60 * 60 # quota has been applied to limit the maximum duration to 1 hour
W = 0.01 # try 0.1, 0.05, 0.01, 0.75
beta = - d ** 2 / log(W)
print(beta)

2814228.242733072


In [ ]:
# Display the first few rows of the origin_dest DataFrame
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0       21    8.434492   11.962262        8.433978       11.961854   
1       23    8.439753   11.972865        8.439239       11.972457   
2       33    8.449757   11.967156        8.449243       11.966748   
3       36    8.451083   11.983468        8.450570       11.983060   
4       44    8.453533   12.005490        8.453019       12.005082   

   origin_lon_max  origin_lat_max  population  duration_seconds  \
0        8.435005       11.962670  156.941269           1142.25   
1        8.440267       11.973273  223.528427            308.07   
2        8.450270       11.967564  233.314590            111.02   
3        8.451597       11.983876  247.544235            933.16   
4        8.454046       12.005898  140.231598           2683.72   

   duration_minutes  distance_km  waterpoint_id       category  dest_lat  \
0         19.037500         1.59             24  Optimal Water  11.96988   
1          5.134500     

In [ ]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [ ]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')

In [ ]:
origin_dest_acc = origin_dest

In [ ]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact. 
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [ ]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [ ]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_minutes,distance_km,waterpoint_id,category,dest_lat,dest_lon,geometry,Weight,Pop_W
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,1142.25,19.037500,1.59,24,Optimal Water,11.96988,8.441043,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",0.629002,98.716366
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,223.528427,308.07,5.134500,0.43,22,Optimal Water,11.97425,8.439657,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ...",0.966838,216.115840
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,233.314590,111.02,1.850333,0.15,67,Optimal Water,11.96606,8.449456,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ...",0.995630,232.294980
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,247.544235,933.16,15.552667,1.30,110,Optimal Water,11.97467,8.455998,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8...",0.733870,181.665315
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,140.231598,2683.72,44.728667,3.73,110,Optimal Water,11.97467,8.455998,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8....",0.077362,10.848549
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103550,159996,8.717747,11.863582,8.717234,11.863175,8.718261,11.863990,4.837119,3586.59,59.776500,4.98,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71725 11.86399, 8.71826 11.86399, ...",0.010348,0.050056
103551,159997,8.717731,11.862767,8.717217,11.862359,8.718245,11.863175,3.875393,3553.37,59.222833,4.94,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71723 11.86317, 8.71824 11.86317, ...",0.011258,0.043631
103552,159998,8.717715,11.861951,8.717201,11.861544,8.718229,11.862359,3.067920,3503.28,58.388000,4.87,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71722 11.86236, 8.71823 11.86236, ...",0.012765,0.039162
103553,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,3468.11,57.801833,4.82,97,Limited Water,11.87966,8.680004,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",0.013927,0.035093


In [ ]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='waterpoint_id')['Pop_W'].sum().reset_index()

In [ ]:
origin_dest_sum

,waterpoint_id,Pop_W
0,1,2358.199666
1,2,1601.714437
2,3,1313.575414
3,4,3066.901273
4,5,1118.906653
...,...,...
390,419,79.734087
391,420,46.816123
392,421,107.112620
393,422,39.898399


In [ ]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='waterpoint_id')

In [ ]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_minutes,distance_km,waterpoint_id,category,dest_lat,dest_lon,geometry,Weight,Pop_W_x,Pop_W_y
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,1142.25,19.037500,1.59,24,Optimal Water,11.96988,8.441043,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",0.629002,98.716366,13335.830810
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,223.528427,308.07,5.134500,0.43,22,Optimal Water,11.97425,8.439657,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ...",0.966838,216.115840,46151.704584
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,233.314590,111.02,1.850333,0.15,67,Optimal Water,11.96606,8.449456,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ...",0.995630,232.294980,1095.195718
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,247.544235,933.16,15.552667,1.30,110,Optimal Water,11.97467,8.455998,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8...",0.733870,181.665315,105159.886330
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,140.231598,2683.72,44.728667,3.73,110,Optimal Water,11.97467,8.455998,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8....",0.077362,10.848549,105159.886330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103550,159996,8.717747,11.863582,8.717234,11.863175,8.718261,11.863990,4.837119,3586.59,59.776500,4.98,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71725 11.86399, 8.71826 11.86399, ...",0.010348,0.050056,4862.145577
103551,159997,8.717731,11.862767,8.717217,11.862359,8.718245,11.863175,3.875393,3553.37,59.222833,4.94,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71723 11.86317, 8.71824 11.86317, ...",0.011258,0.043631,4862.145577
103552,159998,8.717715,11.861951,8.717201,11.861544,8.718229,11.862359,3.067920,3503.28,58.388000,4.87,97,Limited Water,11.87966,8.680004,"POLYGON ((8.71722 11.86236, 8.71823 11.86236, ...",0.012765,0.039162,4862.145577
103553,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,3468.11,57.801833,4.82,97,Limited Water,11.87966,8.680004,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",0.013927,0.035093,4862.145577


In [ ]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [ ]:
supply_map = {
    'Optimal Water': 1.0,   
    'Moderate Water': 0.5,  
    'Limited Water': 0.2    
}

In [ ]:
# Map the supply values to the categories
origin_dest_acc['supply'] = origin_dest_acc['category'].map(supply_map)

In [ ]:
# Calculate the supply-demand ratio
# The supply-demand ratio is calculated by dividing the supply by the Pop_W_S (Population Weight Sum).
# This ratio indicates how well the supply meets the demand in each grid cell.
# A ratio greater than 1 indicates that the supply exceeds the demand, while a ratio less
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_11156/2965326480.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [ ]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [ ]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [ ]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [ ]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_minutes,...,dest_lon,geometry,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,156.941269,1142.25,19.037500,...,8.441043,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",0.629002,98.716366,13335.830810,1.0,0.000075,4.716631e-05,1.032288e-04,0.002805
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,223.528427,308.07,5.134500,...,8.439657,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ...",0.966838,216.115840,46151.704584,1.0,0.000022,2.094913e-05,5.042046e-05,0.001370
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,233.314590,111.02,1.850333,...,8.449456,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ...",0.995630,232.294980,1095.195718,1.0,0.000913,9.090886e-04,9.602116e-04,0.026098
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,247.544235,933.16,15.552667,...,8.455998,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8...",0.733870,181.665315,105159.886330,1.0,0.000010,6.978613e-06,1.174146e-05,0.000319
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,140.231598,2683.72,44.728667,...,8.455998,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8....",0.077362,10.848549,105159.886330,1.0,0.000010,7.356575e-07,9.050656e-07,0.000024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103550,159996,8.717747,11.863582,8.717234,11.863175,8.718261,11.863990,4.837119,3586.59,59.776500,...,8.680004,"POLYGON ((8.71725 11.86399, 8.71826 11.86399, ...",0.010348,0.050056,4862.145577,0.2,0.000041,4.256713e-07,4.256713e-07,0.000011
103551,159997,8.717731,11.862767,8.717217,11.862359,8.718245,11.863175,3.875393,3553.37,59.222833,...,8.680004,"POLYGON ((8.71723 11.86317, 8.71824 11.86317, ...",0.011258,0.043631,4862.145577,0.2,0.000041,4.631030e-07,1.846402e-06,0.000050
103552,159998,8.717715,11.861951,8.717201,11.861544,8.718229,11.862359,3.067920,3503.28,58.388000,...,8.680004,"POLYGON ((8.71722 11.86236, 8.71823 11.86236, ...",0.012765,0.039162,4862.145577,0.2,0.000041,5.250797e-07,2.095389e-06,0.000057
103553,159999,8.717699,11.861136,8.717185,11.860728,8.718212,11.861544,2.519796,3468.11,57.801833,...,8.680004,"POLYGON ((8.7172 11.86154, 8.71821 11.86154, 8...",0.013927,0.035093,4862.145577,0.2,0.000041,5.728780e-07,2.287536e-06,0.000062


In [ ]:
# Find the maximum value of the Accessibility_standard
max(origin_dest_acc.Accessibility_standard)

0.9999999999999999

In [ ]:
# Create a GeoDataFrame from the origin_dest_acc DataFrame
# Ensure the geometry is correctly created from the origin coordinates
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_closest", driver="GPKG")

## 4. Grouping by grid ID to prepare the final output file

In [5]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_closest.gpkg')

In [6]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [7]:
# Remove duplicates based on the specified columns
# This ensures that each grid cell is unique in the results_grid DataFrame
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])

In [8]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'drinking_water_deprivation_access.gpkg'
results_grid.to_file(output_gpkg_path, driver='GPKG')

In [9]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,0.002805,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ..."
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,0.001370,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ..."
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,0.026098,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ..."
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,0.000319,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8..."
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,0.000024,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8...."
...,...,...,...,...,...,...,...,...,...
103533,159454,8.715789,11.866844,8.715276,11.866437,8.716303,11.867252,0.000011,"POLYGON ((8.71529 11.86725, 8.7163 11.86725, 8..."
103542,159463,8.715644,11.859505,8.715131,11.859097,8.716158,11.859913,0.000012,"POLYGON ((8.71515 11.85991, 8.71616 11.85991, ..."
103543,159724,8.716768,11.865213,8.716255,11.864806,8.717282,11.865621,0.000011,"POLYGON ((8.71627 11.86562, 8.71728 11.86562, ..."
103549,159730,8.716672,11.860320,8.716158,11.859913,8.717185,11.860728,0.000012,"POLYGON ((8.71617 11.86073, 8.71719 11.86073, ..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  


In [10]:
# Assign deprivation levels based on the Accessibility_standard values
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] >= 0, 'result'] = 2 #(high deprivation)
results_grid.loc[results_grid['Accessibility_standard'] > 0.0004202, 'result'] = 1 #(medium deprivation)
results_grid.loc[results_grid['Accessibility_standard'] > 0.0130206, 'result'] = 0 #(low deprivation)

### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [13]:
#We set Dorayi Karama boundary as our focus area
from pyproj import CRS

def get_utm_crs(geo):
    lon, lat = geo.unary_union.centroid.x, geo.unary_union.centroid.y
    zone = int((lon + 180) // 6) + 1
    if lat >= 0:
        epsg_code = 32600 + zone  # Northern hemisphere
    else:
        epsg_code = 32700 + zone  # Southern hemisphere
    return CRS.from_epsg(epsg_code)


# 1. Read community boundary
dorayi_karama = gpd.read_file(data_inputs + "Dorayi_Karama.gpkg")

# Automatically choose the correct UTM CRS
utm_crs = get_utm_crs(dorayi_karama)

# Reproject boundary to UTM
dorayi_karama_utm = dorayi_karama.to_crs(utm_crs)

# Create 2km buffer
focus_area = dorayi_karama_utm.buffer(2000, join_style=2).unary_union


# Reproject grid to same UTM CRS
results_grid_utm = results_grid.to_crs(utm_crs)

# 3. Check intersection
results_grid_utm["focused"] = results_grid_utm.intersects(focus_area).astype(int)

# 4. Reproject back to original CRS (e.g., EPSG:4326)
results_grid = results_grid_utm.to_crs("EPSG:4326")

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_36473/4267755858.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lon, lat = geo.unary_union.centroid.x, geo.unary_union.centroid.y
/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_36473/4267755858.py:24: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  focus_area = dorayi_karama_utm.buffer(2000, join_style=2).unary_union


In [14]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry,result,focused
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,0.002805,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",1,0
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,0.001370,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ...",1,0
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,0.026098,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ...",0,1
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,0.000319,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8...",2,0
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,0.000024,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8....",2,0
...,...,...,...,...,...,...,...,...,...,...,...
103533,159454,8.715789,11.866844,8.715276,11.866437,8.716303,11.867252,0.000011,"POLYGON ((8.71529 11.86725, 8.7163 11.86725, 8...",2,0
103542,159463,8.715644,11.859505,8.715131,11.859097,8.716158,11.859913,0.000012,"POLYGON ((8.71515 11.85991, 8.71616 11.85991, ...",2,0
103543,159724,8.716768,11.865213,8.716255,11.864806,8.717282,11.865621,0.000011,"POLYGON ((8.71627 11.86562, 8.71728 11.86562, ...",2,0
103549,159730,8.716672,11.860320,8.716158,11.859913,8.717185,11.860728,0.000012,"POLYGON ((8.71617 11.86073, 8.71719 11.86073, ...",2,0


In [15]:
# Remove rows where the result is -1 (no deprivation)
# This ensures that only relevant results are kept in the results_grid DataFrame
results_grid = results_grid.loc[results_grid['result'] != -1]

In [16]:
# Rename columns for clarity
# This makes the column names more descriptive and easier to understand
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [17]:
# Remove the 'geometry' column if it is not needed
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
0,21,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,0.002805,"POLYGON ((8.43399 11.96267, 8.43501 11.96267, ...",1,0
1,23,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,0.001370,"POLYGON ((8.43926 11.97327, 8.44027 11.97327, ...",1,0
2,33,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,0.026098,"POLYGON ((8.44926 11.96756, 8.45027 11.96756, ...",0,1
3,36,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,0.000319,"POLYGON ((8.45059 11.98388, 8.4516 11.98388, 8...",2,0
4,44,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,0.000024,"POLYGON ((8.45303 12.0059, 8.45405 12.0059, 8....",2,0
...,...,...,...,...,...,...,...,...,...,...,...
103533,159454,8.715789,11.866844,8.715276,11.866437,8.716303,11.867252,0.000011,"POLYGON ((8.71529 11.86725, 8.7163 11.86725, 8...",2,0
103542,159463,8.715644,11.859505,8.715131,11.859097,8.716158,11.859913,0.000012,"POLYGON ((8.71515 11.85991, 8.71616 11.85991, ...",2,0
103543,159724,8.716768,11.865213,8.716255,11.864806,8.717282,11.865621,0.000011,"POLYGON ((8.71627 11.86562, 8.71728 11.86562, ...",2,0
103549,159730,8.716672,11.860320,8.716158,11.859913,8.717185,11.860728,0.000012,"POLYGON ((8.71617 11.86073, 8.71719 11.86073, ...",2,0


In [ ]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'drinking-water-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='drinking-water-access-class', driver='GPKG')

In [19]:
print(results_grid.columns)


Index(['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max',
       'lat_max', 'Accessibility_standard', 'geometry', 'result', 'focused'],
      dtype='object')


In [25]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)

In [26]:
# Display the results table
results_table

,longitude,latitude,lon_min,lat_min,lon_max,lat_max,result,focused
0,8.434492,11.962262,8.433978,11.961854,8.435005,11.962670,1,0
1,8.439753,11.972865,8.439239,11.972457,8.440267,11.973273,1,0
2,8.449757,11.967156,8.449243,11.966748,8.450270,11.967564,0,1
3,8.451083,11.983468,8.450570,11.983060,8.451597,11.983876,2,0
4,8.453533,12.005490,8.453019,12.005082,8.454046,12.005898,2,0
...,...,...,...,...,...,...,...,...
103533,8.715789,11.866844,8.715276,11.866437,8.716303,11.867252,2,0
103542,8.715644,11.859505,8.715131,11.859097,8.716158,11.859913,2,0
103543,8.716768,11.865213,8.716255,11.864806,8.717282,11.865621,2,0
103549,8.716672,11.860320,8.716158,11.859913,8.717185,11.860728,2,0
